In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'Step0_Data').is_dir() and (candidate / 'Step2_WDI_Features').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate standalone release root from the current working directory')

release_root = find_release_root(cwd)
step_root = release_root / 'Step3_BuildLongTable'
input_path = release_root / 'Step2_WDI_Features' / '2_Results' / '1_imputed_features_data.csv'
output_path = step_root / '2_Results' / '0_LongTable_SetABC.csv'
incomegroup_override_by_iso3 = {'ETH': 'Low income'}
tasks = {'AW_t': 'Is_AW', 'CDW_t': 'Is_CDW', 'IW_t': 'Is_IW', 'MSW_t': 'Is_MSW'}
base_proxy_cols = [
    'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'SP.POP.TOTL', 'EN.POP.DNST',
    'SP.URB.TOTL.IN.ZS', 'SP.URB.GROW', 'NV.IND.TOTL.ZS', 'NV.SRV.TOTL.ZS', 'EN.GHG.CO2.MT.CE.AR5',
]

In [ ]:
df_wide = pd.read_csv(input_path)
if 'IncomeGroup' in df_wide.columns:
    df_wide['IncomeGroup'] = df_wide['IncomeGroup'].fillna(df_wide['Country Code'].map(incomegroup_override_by_iso3))
df_wide.shape

In [ ]:
target_cols = list(tasks.keys())
base_features = [col for col in df_wide.columns if col not in target_cols]
long_chunks = []
for target_col, flag_col in tasks.items():
    chunk = df_wide.dropna(subset=[target_col]).copy()
    chunk = chunk[base_features + [target_col]].copy()
    chunk['Waste_Generation_t'] = chunk[target_col]
    chunk = chunk.drop(columns=[target_col])
    for current_target, current_flag in tasks.items():
        chunk[current_flag] = 1 if current_target == target_col else 0
    long_chunks.append(chunk)
df_long = pd.concat(long_chunks, ignore_index=True)
df_long = df_long.sort_values(['Country Code', 'Year']).reset_index(drop=True)
df_long.shape

In [ ]:
for col in base_proxy_cols:
    if col not in df_long.columns:
        continue
    base = pd.to_numeric(df_long[col], errors='coerce')
    df_long[f'Proxy_{col}_log1p'] = np.log1p(base.clip(lower=0.0))
    for flag_col in ['Is_AW', 'Is_CDW', 'Is_IW', 'Is_MSW']:
        flag_short = flag_col.replace('Is_', '')
        df_long[f'Proxy_{col}_x_{flag_short}'] = base * pd.to_numeric(df_long[flag_col], errors='coerce')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_long.to_csv(output_path, index=False)
output_path